In [1]:
import langchain

In [2]:
from langgraph.graph import StateGraph, START, END, add_messages
from langgraph.types import Command, interrupt
from typing import Annotated, List, TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

In [3]:
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

In [ ]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field

# Your LLM (e.g., from langchain_openai)
model = llm

# Step 1: Define output structure
class Joke(BaseModel):
    setup: str = Field(description="Question to set up a joke")
    punchline: str = Field(description="Answer to resolve the joke")
    clarification_needed: bool = Field(
        default=False, description="Set to true if the model needs more information to proceed"
    )

# Step 2: Create the output parser
parser = JsonOutputParser(pydantic_object=Joke)

# Step 3: Create a well-instructed prompt template
prompt = PromptTemplate(
    template="""
You are a joke-telling assistant.

Your job is to fill in a structured response that has three fields:
- setup (string)
- punchline (string)
- clarification_needed (boolean)

IMPORTANT RULES:
1. Only write a joke (setup and punchline) if the topic is clearly and explicitly stated in the query.
2. DO NOT GUESS the topic. If you are not sure, DO NOT make assumptions.
3. If the topic is unclear or missing, ask for clarification in the `setup`, leave the `punchline` empty, and set `clarification_needed` to true.

{format_instructions}

Here is the user query: {query}
""",
    input_variables=["query"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

# Step 4: Compose the chain
chain = prompt | model | parser

# Step 5: Now invoke with a real user query (missing topic)
result = chain.invoke({
    "query": "Tell me a joke about that thing we discussed."
})
print(result)


{'setup': "I'm sorry, but what was the topic we discussed? I need to know the topic to tell a joke.", 'punchline': '', 'clarification_needed': True}


In [ ]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate

In [36]:
from pydantic import BaseModel, Field

class ThinkerOutput(BaseModel):
    clarification_needed: bool = Field(description="True if clarification is still needed")
    output: str = Field(description="Either a clarification question or the structured project summary")


In [37]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder


In [42]:
class state(TypedDict):
    messages: Annotated[List[str],add_messages]
    clarification_count: int
    clarification_needed: bool

In [ ]:

thinker_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a senior product analyst and technical solution designer.

Your job is to clarify and refine software project ideas through a structured conversation.

Clarification round: {clarification_count}/3

Instructions:
- If the user's idea is unclear, respond ONLY with a **clarification question** that will help you understand the idea better.
- Set `clarification_needed` to true.
- If the idea is sufficiently clear or after 3 rounds, respond with a **structured project summary** covering:
    - Project goals
    - Target users
    - Key components or features
- Set `clarification_needed` to false in this case.

Return your response as a JSON object with exactly two fields:
- `clarification_needed`: true or false
- `output`: a string containing either the clarification question or the project summary

Do NOT output anything outside the JSON.

{format_instructions}

Conversation so far:
{messages}

User's latest input is the last user message in the conversation.
"""
    ),
    MessagesPlaceholder(variable_name="messages"),
])

In [ ]:
# from langchain_core.output_parsers import JsonOutputParser

# parser = JsonOutputParser(pydantic_object=ThinkerOutput)
# # 
# chain = thinker_prompt | llm | parser


In [4]:
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import JsonOutputParser

# Define the structured output model
class ThinkerOutput(BaseModel):
    clarification_needed: bool = Field(description="True if clarification is still needed")
    output: str = Field(description="Clarification question or project summary")

# Create the chat prompt template with instructions
thinker_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a senior product analyst and technical solution designer.

Your job is to clarify software projects through structured conversation, then deliver a comprehensive project blueprint.

Clarification round: {clarification_count}/3

CLARIFICATION PHASE (clarification_needed = true):
- Ask ONE targeted question about: target users, core functionality, technical needs, or business goals
- Keep questions specific and actionable

FINAL SUMMARY PHASE (clarification_needed = false):
Provide a detailed project blueprint covering:

1. **PROJECT OVERVIEW** - Title, description, objectives, success metrics
2. **TARGET USERS** - User groups, pain points, user journeys  
3. **CORE FEATURES** - Feature breakdown with priorities, user stories
4. **TECHNICAL ARCHITECTURE** - Tech stack, system design, integrations, scalability
5. **IMPLEMENTATION PLAN** - Development phases, MVP scope, timelines, resources
6. **RISKS & MITIGATION** - Technical/business risks and solutions
7. **SUCCESS METRICS** - KPIs and measurement criteria

Make the final summary comprehensive (800-1200 words) and actionable for development planning.

OUTPUT: Return ONLY JSON with two fields:
- `clarification_needed`: true/false
- `output`: clarification question OR detailed project blueprint

{format_instructions}

{messages}
"""
    ),
    MessagesPlaceholder(variable_name="messages"),
])
# Setup the parser
parser = JsonOutputParser(pydantic_object=ThinkerOutput)

# Compose the full chain: prompt -> LLM model -> parser
chain = thinker_prompt | llm | parser

# Initialize conversation state
state = {
    "messages": [],
    "clarification_count": 0,
    "clarification_needed": True,
}

def run_thinker(user_input: str):
    # Append new user input to conversation messages
    state["messages"].append({"role": "user", "content": user_input})

    # Call the chain with current state
    response = chain.invoke({
        "messages": state["messages"],
        "clarification_count": state["clarification_count"],
        "format_instructions": parser.get_format_instructions(),
    })
    response = ThinkerOutput(**response)
    # Update state based on LLM response
    if response.clarification_needed :
        state["clarification_needed"] = True
        state["clarification_count"] += 1
        # Add assistant's clarification question to messages
        state["messages"].append({"role": "assistant", "content": response.output})
    else:
        state["clarification_needed"] = False
        # Add the final structured summary to messages
        state["messages"].append({"role": "assistant", "content": response.output})

    return response
    

# Example usage:
user_idea = input("Enter the project idea : ")
result = run_thinker(user_idea)
print(result)


clarification_needed=True output='To ensure the website effectively serves its users, could you describe the key functionalities you envision for the library management system, such as catalog search, user account management, or inventory tracking?'


In [5]:
user_idea = input("Enter the clarification_question ")
result = run_thinker(user_idea)
print(result.output)
print(state["messages"])
user_idea = input("Enter the clarification_question ")
result = run_thinker(user_idea)
print(result.output)
print(state["messages"])
user_idea = input("Enter the clarification_question ")
result = run_thinker(user_idea)
print(result.output)
print(state["messages"])

To clarify the scope, are there any specific types of users (e.g., librarians, students, faculty) who will have different roles or access levels within the library management system?
[{'role': 'user', 'content': 'Make a website for library management system'}, {'role': 'assistant', 'content': 'To ensure the website effectively serves its users, could you describe the key functionalities you envision for the library management system, such as catalog search, user account management, or inventory tracking?'}, {'role': 'user', 'content': 'All the features that you said right now '}, {'role': 'assistant', 'content': 'To clarify the scope, are there any specific types of users (e.g., librarians, students, faculty) who will have different roles or access levels within the library management system?'}]
## Library Management System Website: Project Blueprint

**1. PROJECT OVERVIEW**

*   **Title:** Library Management System Website
*   **Description:** A comprehensive web-based platform to man